# Advanced Problems: Function Introspection

This notebook contains advanced exercises with full solutions for Python function introspection.

Topics covered:

- Function attributes: `__name__`, `__qualname__`, `__doc__`, `__annotations__`, `__dict__`
- Code object attributes: `co_argcount`, `co_posonlyargcount`, `co_kwonlyargcount`, `co_varnames`, `co_freevars`, `co_cellvars`
- `inspect.signature`, `Parameter.kind`, `bind`, and `bind_partial`
- Bound methods, class methods, static methods, and callable objects
- Closures and decorators
- Preserving metadata with `functools.wraps`
- Building practical introspection utilities


In [1]:
import inspect
import functools
from pprint import pprint


## Problem 1 — Build a Function Metadata Extractor

### Problem

Write a function `describe_callable(obj)` that returns a dictionary describing a callable.

Include name, qualified name, module, docstring, annotations, custom attributes, routine type checks, and signature.


### Solution


In [2]:
def describe_callable(obj):
    if not callable(obj):
        raise TypeError(f'{obj!r} is not callable')

    try:
        sig = str(inspect.signature(obj))
    except (TypeError, ValueError):
        sig = None

    return {
        'name': getattr(obj, '__name__', type(obj).__name__),
        'qualname': getattr(obj, '__qualname__', type(obj).__qualname__),
        'module': getattr(obj, '__module__', type(obj).__module__),
        'doc': inspect.getdoc(obj),
        'annotations': dict(getattr(obj, '__annotations__', {})),
        'custom_attributes': dict(getattr(obj, '__dict__', {})),
        'is_function': inspect.isfunction(obj),
        'is_method': inspect.ismethod(obj),
        'is_builtin': inspect.isbuiltin(obj),
        'is_routine': inspect.isroutine(obj),
        'signature': sig,
    }


def scale(x: int, factor: int = 2) -> int:
    '''Scale x by a factor.'''
    return x * factor

scale.category = 'math helper'

class Demo:
    def method(self, value: str) -> str:
        return f'value={value}'

class CallableDemo:
    def __call__(self, a, b=1):
        return a + b

examples = [scale, lambda x, y=10: x + y, len, Demo().method, CallableDemo()]

for item in examples:
    print('\n---', item, '---')
    pprint(describe_callable(item))



--- <function scale at 0x00000224566527A0> ---
{'annotations': {'factor': <class 'int'>,
                 'return': <class 'int'>,
                 'x': <class 'int'>},
 'custom_attributes': {'category': 'math helper'},
 'doc': 'Scale x by a factor.',
 'is_builtin': False,
 'is_function': True,
 'is_method': False,
 'is_routine': True,
 'module': '__main__',
 'name': 'scale',
 'qualname': 'scale',
 'signature': '(x: int, factor: int = 2) -> int'}

--- <function <lambda> at 0x0000022456652160> ---
{'annotations': {},
 'custom_attributes': {},
 'doc': None,
 'is_builtin': False,
 'is_function': True,
 'is_method': False,
 'is_routine': True,
 'module': '__main__',
 'name': '<lambda>',
 'qualname': '<lambda>',
 'signature': '(x, y=10)'}

--- <built-in function len> ---
{'annotations': {},
 'custom_attributes': {},
 'doc': 'Return the number of items in a container.',
 'is_builtin': True,
 'is_function': False,
 'is_method': False,
 'is_routine': True,
 'module': 'builtins',
 'name': 'len

## Problem 2 — Reconstruct Argument Layout from a Code Object

### Problem

Write `code_layout(fn)` that uses `fn.__code__` to report positional-only parameters, positional-or-keyword parameters, keyword-only parameters, `*args`, `**kwargs`, local-only variables, free variables, and cell variables.


### Solution


In [3]:
def code_layout(fn):
    if not inspect.isfunction(fn):
        raise TypeError('code_layout expects a Python function')

    code = fn.__code__
    varnames = code.co_varnames

    posonly_count = getattr(code, 'co_posonlyargcount', 0)
    pos_or_kw_count = code.co_argcount - posonly_count
    kwonly_count = code.co_kwonlyargcount

    index = 0
    posonly = varnames[index:index + posonly_count]
    index += posonly_count

    pos_or_kw = varnames[index:index + pos_or_kw_count]
    index += pos_or_kw_count

    kwonly = varnames[index:index + kwonly_count]
    index += kwonly_count

    has_varargs = bool(code.co_flags & inspect.CO_VARARGS)
    has_varkw = bool(code.co_flags & inspect.CO_VARKEYWORDS)

    varargs = varnames[index] if has_varargs else None
    if has_varargs:
        index += 1

    varkw = varnames[index] if has_varkw else None
    if has_varkw:
        index += 1

    return {
        'positional_only': posonly,
        'positional_or_keyword': pos_or_kw,
        'keyword_only': kwonly,
        'varargs': varargs,
        'varkw': varkw,
        'locals_only': varnames[index:code.co_nlocals],
        'freevars': code.co_freevars,
        'cellvars': code.co_cellvars,
    }


def outer(rate):
    prefix = 'result'

    def inner(a, b, /, c=10, *args, flag=True, **kwargs):
        local_total = a + b + c + sum(args)
        if flag:
            return f'{prefix}: {local_total * rate}'
        return kwargs

    return inner

fn = outer(3)
pprint(code_layout(fn))
print(inspect.signature(fn))


{'cellvars': (),
 'freevars': ('prefix', 'rate'),
 'keyword_only': ('flag',),
 'locals_only': ('local_total',),
 'positional_only': ('a', 'b'),
 'positional_or_keyword': ('c',),
 'varargs': 'args',
 'varkw': 'kwargs'}
(a, b, /, c=10, *args, flag=True, **kwargs)


## Problem 3 — Classify Parameters with `inspect.signature`

### Problem

Write `signature_table(fn)` that returns one dictionary per parameter with name, kind, default information, and annotation information.


### Solution


In [4]:
def signature_table(fn):
    sig = inspect.signature(fn)
    rows = []

    for param in sig.parameters.values():
        rows.append({
            'name': param.name,
            'kind': str(param.kind),
            'has_default': param.default is not inspect.Parameter.empty,
            'default': None if param.default is inspect.Parameter.empty else param.default,
            'has_annotation': param.annotation is not inspect.Parameter.empty,
            'annotation': None if param.annotation is inspect.Parameter.empty else param.annotation,
        })

    return rows


def complicated(a, b: int, /, c: str = 'x', *items: float, debug: bool, limit=100, **options) -> tuple:
    return a, b, c, items, debug, limit, options

pprint(signature_table(complicated))
print('Return annotation:', inspect.signature(complicated).return_annotation)


[{'annotation': None,
  'default': None,
  'has_annotation': False,
  'has_default': False,
  'kind': 'POSITIONAL_ONLY',
  'name': 'a'},
 {'annotation': <class 'int'>,
  'default': None,
  'has_annotation': True,
  'has_default': False,
  'kind': 'POSITIONAL_ONLY',
  'name': 'b'},
 {'annotation': <class 'str'>,
  'default': 'x',
  'has_annotation': True,
  'has_default': True,
  'kind': 'POSITIONAL_OR_KEYWORD',
  'name': 'c'},
 {'annotation': <class 'float'>,
  'default': None,
  'has_annotation': True,
  'has_default': False,
  'kind': 'VAR_POSITIONAL',
  'name': 'items'},
 {'annotation': <class 'bool'>,
  'default': None,
  'has_annotation': True,
  'has_default': False,
  'kind': 'KEYWORD_ONLY',
  'name': 'debug'},
 {'annotation': None,
  'default': 100,
  'has_annotation': False,
  'has_default': True,
  'kind': 'KEYWORD_ONLY',
  'name': 'limit'},
 {'annotation': None,
  'default': None,
  'has_annotation': False,
  'has_default': False,
  'kind': 'VAR_KEYWORD',
  'name': 'options'

## Problem 4 — Validate Calls Without Calling the Function

### Problem

Create `validate_call(fn, *args, **kwargs)` using `inspect.signature(fn).bind(...)`. It should report whether a call would be valid without actually executing the callable.


### Solution


In [5]:
def validate_call(fn, *args, **kwargs):
    sig = inspect.signature(fn)

    try:
        bound = sig.bind(*args, **kwargs)
    except TypeError as ex:
        return {'ok': False, 'error': str(ex)}

    return {'ok': True, 'bound': dict(bound.arguments)}


def api(endpoint, /, method='GET', *, timeout, retries=3, **headers):
    raise RuntimeError('This should not run during validation')


tests = [
    (('/users',), {'timeout': 10}),
    ((), {'endpoint': '/users', 'timeout': 10}),
    (('/users', 'POST'), {'timeout': 10, 'Authorization': 'token'}),
    (('/users',), {}),
    (('/users',), {'timeout': 10, 'method': 'PATCH'}),
]

for args, kwargs in tests:
    print(args, kwargs, '=>')
    pprint(validate_call(api, *args, **kwargs))
    print()


('/users',) {'timeout': 10} =>
{'bound': {'endpoint': '/users', 'timeout': 10}, 'ok': True}

() {'endpoint': '/users', 'timeout': 10} =>
{'error': "missing a required positional-only argument: 'endpoint'",
 'ok': False}

('/users', 'POST') {'timeout': 10, 'Authorization': 'token'} =>
{'bound': {'endpoint': '/users',
           'headers': {'Authorization': 'token'},
           'method': 'POST',
           'timeout': 10},
 'ok': True}

('/users',) {} =>
{'error': "missing a required argument: 'timeout'", 'ok': False}

('/users',) {'timeout': 10, 'method': 'PATCH'} =>
{'bound': {'endpoint': '/users', 'method': 'PATCH', 'timeout': 10}, 'ok': True}



## Problem 5 — Apply Defaults to Bound Arguments

### Problem

Write `normalized_call_arguments(fn, *args, **kwargs)` that validates a call, applies defaults, and returns the final argument mapping.


### Solution


In [6]:
def normalized_call_arguments(fn, *args, **kwargs):
    sig = inspect.signature(fn)
    bound = sig.bind(*args, **kwargs)
    bound.apply_defaults()
    return dict(bound.arguments)


def task(name, count=1, *tags, urgent=False, **metadata):
    pass

print(normalized_call_arguments(task, 'backup', 'nightly', urgent=True, owner='ops'))
print(normalized_call_arguments(task, 'cleanup'))


{'name': 'backup', 'count': 'nightly', 'tags': (), 'urgent': True, 'metadata': {'owner': 'ops'}}
{'name': 'cleanup', 'count': 1, 'tags': (), 'urgent': False, 'metadata': {}}


## Problem 6 — Detect Bound Methods, Static Methods, and Class Methods

### Problem

Create `classify_class_attribute(cls, attr_name)`. Use `inspect.getattr_static` so descriptors are not triggered automatically.


### Solution


In [7]:
def classify_class_attribute(cls, attr_name):
    try:
        raw = inspect.getattr_static(cls, attr_name)
    except AttributeError:
        return 'missing'

    if isinstance(raw, staticmethod):
        return 'static method'

    if isinstance(raw, classmethod):
        return 'class method'

    if inspect.isfunction(raw):
        return 'instance function'

    return 'plain attribute'


class Service:
    version = '1.0'

    def instance_method(self):
        pass

    @classmethod
    def class_method(cls):
        pass

    @staticmethod
    def static_method():
        pass

for name in ['version', 'instance_method', 'class_method', 'static_method', 'unknown']:
    print(name, '=>', classify_class_attribute(Service, name))

s = Service()
print('\nRuntime binding behavior:')
print('Service.instance_method:', inspect.isfunction(Service.instance_method), inspect.ismethod(Service.instance_method))
print('s.instance_method:', inspect.isfunction(s.instance_method), inspect.ismethod(s.instance_method))
print('Service.class_method:', inspect.isfunction(Service.class_method), inspect.ismethod(Service.class_method))
print('Service.static_method:', inspect.isfunction(Service.static_method), inspect.ismethod(Service.static_method))


version => plain attribute
instance_method => instance function
class_method => class method
static_method => static method
unknown => missing

Runtime binding behavior:
Service.instance_method: True False
s.instance_method: False True
Service.class_method: False True
Service.static_method: True False


## Problem 7 — Introspect Closures

### Problem

Write `closure_vars(fn)` that returns a dictionary mapping each free variable name to the value captured in the closure.


### Solution


In [8]:
def closure_vars(fn):
    if not inspect.isfunction(fn):
        raise TypeError('closure_vars expects a Python function')

    names = fn.__code__.co_freevars
    cells = fn.__closure__ or ()

    return {name: cell.cell_contents for name, cell in zip(names, cells)}


def make_multiplier(factor):
    offset = 1

    def multiply(x):
        return x * factor + offset

    return multiply

times_10_plus_1 = make_multiplier(10)

print(times_10_plus_1(5))
pprint(closure_vars(times_10_plus_1))
print('co_freevars:', times_10_plus_1.__code__.co_freevars)
print('__closure__:', times_10_plus_1.__closure__)


51
{'factor': 10, 'offset': 1}
co_freevars: ('factor', 'offset')
__closure__: (<cell at 0x00000224566338E0: int object at 0x00007FFC836E74C8>, <cell at 0x0000022456633280: int object at 0x00007FFC836E73A8>)


## Problem 8 — Preserve Function Metadata in a Decorator

### Problem

Write a bad decorator that does not preserve metadata, and a good decorator that uses `functools.wraps`. Compare `__name__`, `__doc__`, `__annotations__`, `inspect.signature`, and `inspect.unwrap`.


### Solution


In [9]:
def bad_timer(fn):
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)
    return wrapper


def good_timer(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        return fn(*args, **kwargs)
    return wrapper


@bad_timer
def bad_add(x: int, y: int = 0) -> int:
    '''Add two integers.'''
    return x + y


@good_timer
def good_add(x: int, y: int = 0) -> int:
    '''Add two integers.'''
    return x + y


for fn in [bad_add, good_add]:
    print('\n---', fn, '---')
    print('__name__:', fn.__name__)
    print('__doc__:', fn.__doc__)
    print('__annotations__:', fn.__annotations__)
    print('signature:', inspect.signature(fn))
    print('unwrapped:', inspect.unwrap(fn))
    print('unwrapped signature:', inspect.signature(inspect.unwrap(fn)))



--- <function bad_timer.<locals>.wrapper at 0x0000022456653BA0> ---
__name__: wrapper
__doc__: None
__annotations__: {}
signature: (*args, **kwargs)
unwrapped: <function bad_timer.<locals>.wrapper at 0x0000022456653BA0>
unwrapped signature: (*args, **kwargs)

--- <function good_add at 0x0000022456653CE0> ---
__name__: good_add
__doc__: Add two integers.
__annotations__: {'x': <class 'int'>, 'y': <class 'int'>, 'return': <class 'int'>}
signature: (x: int, y: int = 0) -> int
unwrapped: <function good_add at 0x0000022456653C40>
unwrapped signature: (x: int, y: int = 0) -> int


## Problem 9 — Generate Documentation from Signatures

### Problem

Build `markdown_api_doc(fn)` that creates Markdown documentation from function introspection data.


### Solution


In [10]:
def format_annotation(value):
    if value is inspect.Signature.empty or value is inspect.Parameter.empty:
        return ''
    if isinstance(value, type):
        return value.__name__
    return repr(value)


def markdown_api_doc(fn):
    sig = inspect.signature(fn)
    name = getattr(fn, '__name__', type(fn).__name__)
    lines = []

    lines.append(f'## `{name}`')
    lines.append('')
    lines.append(f'```python\n{name}{sig}\n```')
    lines.append('')

    doc = inspect.getdoc(fn)
    if doc:
        lines.append(inspect.cleandoc(doc))
        lines.append('')

    lines.append('| Parameter | Kind | Default | Annotation |')
    lines.append('|---|---|---:|---|')

    for param in sig.parameters.values():
        default = '' if param.default is inspect.Parameter.empty else repr(param.default)
        annotation = format_annotation(param.annotation)
        lines.append(f'| `{param.name}` | `{param.kind}` | `{default}` | `{annotation}` |')

    lines.append('')
    lines.append(f'**Returns:** `{format_annotation(sig.return_annotation) or "None specified"}`')
    return '\n'.join(lines)


def send_email(to: str, subject: str, /, body: str = '', *, cc=None, urgent: bool = False) -> bool:
    '''Send an email message.

    This example does not actually send anything.
    '''
    return True

print(markdown_api_doc(send_email))


## `send_email`

```python
send_email(to: str, subject: str, /, body: str = '', *, cc=None, urgent: bool = False) -> bool
```

Send an email message.

This example does not actually send anything.

| Parameter | Kind | Default | Annotation |
|---|---|---:|---|
| `to` | `POSITIONAL_ONLY` | `` | `str` |
| `subject` | `POSITIONAL_ONLY` | `` | `str` |
| `body` | `POSITIONAL_OR_KEYWORD` | `''` | `str` |
| `cc` | `KEYWORD_ONLY` | `None` | `` |
| `urgent` | `KEYWORD_ONLY` | `False` | `bool` |

**Returns:** `bool`


## Problem 10 — Build a Runtime Type-Checking Decorator from Annotations

### Problem

Create `enforce_annotations(fn)`. It should preserve metadata, bind arguments, apply defaults, validate annotated arguments, and validate the return annotation when it is a real type.


### Solution


In [11]:
def enforce_annotations(fn):
    sig = inspect.signature(fn)

    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        bound.apply_defaults()

        for name, value in bound.arguments.items():
            annotation = sig.parameters[name].annotation
            if isinstance(annotation, type) and not isinstance(value, annotation):
                raise TypeError(
                    f'Argument {name!r} expected {annotation.__name__}, '
                    f'got {type(value).__name__}'
                )

        result = fn(*args, **kwargs)

        if isinstance(sig.return_annotation, type) and not isinstance(result, sig.return_annotation):
            raise TypeError(
                f'Return value expected {sig.return_annotation.__name__}, '
                f'got {type(result).__name__}'
            )

        return result

    return wrapper


@enforce_annotations
def repeat(text: str, times: int = 2) -> str:
    return text * times

print(repeat('ha', 3))
print('__name__:', repeat.__name__)
print('signature:', inspect.signature(repeat))

try:
    repeat('ha', '3')
except TypeError as ex:
    print('TypeError:', ex)


@enforce_annotations
def broken() -> int:
    return 'not an int'

try:
    broken()
except TypeError as ex:
    print('TypeError:', ex)


hahaha
__name__: repeat
signature: (text: str, times: int = 2) -> str
TypeError: Argument 'times' expected int, got str
TypeError: Return value expected int, got str


## Problem 11 — Introspect Callable Objects

### Problem

Write `callable_object_report(obj)` that reports whether an object is callable, its type name, the signature of the object itself, the signature of `type(obj).__call__`, and whether `obj.__call__` is bound.


### Solution


In [12]:
def callable_object_report(obj):
    report = {
        'callable': callable(obj),
        'type_name': type(obj).__name__,
    }

    if callable(obj):
        report['object_signature'] = str(inspect.signature(obj))
        report['type_call_signature'] = str(inspect.signature(type(obj).__call__))
        report['obj_call_is_method'] = inspect.ismethod(obj.__call__)

    return report


class Power:
    def __init__(self, exponent):
        self.exponent = exponent

    def __call__(self, base: float, *, modulo=None):
        value = base ** self.exponent
        return value if modulo is None else value % modulo

square = Power(2)

pprint(callable_object_report(square))
print('square(5):', square(5))
print('square(5, modulo=7):', square(5, modulo=7))


{'callable': True,
 'obj_call_is_method': True,
 'object_signature': '(base: float, *, modulo=None)',
 'type_call_signature': '(self, base: float, *, modulo=None)',
 'type_name': 'Power'}
square(5): 25
square(5, modulo=7): 4


## Problem 12 — Detect Suspicious Mutable Defaults

### Problem

Write `find_mutable_defaults(fn)` that detects defaults that are instances of common mutable built-in types: `list`, `dict`, `set`, and `bytearray`.


### Solution


In [13]:
MUTABLE_DEFAULT_TYPES = (list, dict, set, bytearray)


def find_mutable_defaults(fn):
    sig = inspect.signature(fn)
    suspicious = {}

    for name, param in sig.parameters.items():
        if param.default is not inspect.Parameter.empty and isinstance(param.default, MUTABLE_DEFAULT_TYPES):
            suspicious[name] = param.default

    return suspicious


def bad_cache(key, cache={}):
    cache[key] = key.upper()
    return cache[key]


def good_cache(key, cache=None):
    if cache is None:
        cache = {}
    cache[key] = key.upper()
    return cache[key]

print('bad_cache:', find_mutable_defaults(bad_cache))
print('good_cache:', find_mutable_defaults(good_cache))

print(bad_cache('a'))
print(bad_cache('b'))
print('Shared default:', bad_cache.__defaults__)


bad_cache: {'cache': {}}
good_cache: {}
A
B
Shared default: ({'a': 'A', 'b': 'B'},)


## Best-Practice Takeaways

- Prefer `inspect.signature()` over manually reading code object attributes for normal call validation.
- Use code object attributes when you specifically need implementation-level details such as local variables, free variables, and cell variables.
- Use `inspect.getattr_static()` when you need to inspect class attributes without triggering descriptor binding.
- Use `functools.wraps()` in decorators so metadata, signatures, and unwrapping remain useful.
- Use `Signature.bind()` and `BoundArguments.apply_defaults()` to normalize function calls safely without executing the function.
- Remember that callable objects, bound methods, static methods, class methods, built-ins, and Python functions expose different introspection surfaces.
